# Law as Code — Prototype Notebook

Interactive prototyping for the DSPy legal-text-to-code pipeline.

## Setup

In [ ]:
import os

from dotenv import load_dotenv
import dspy

from oll_law_as_code.pipeline import LegalToCode, LegalTransformer
from oll_law_as_code.runner import run_generated_code
from oll_law_as_code.persona_runner import run_persona_tests
from oll_law_as_code.personas import ALL_PERSONAS

load_dotenv()

In [ ]:
from oll_law_as_code.tracking import setup_tracking

experiment = setup_tracking()
print(f"MLflow tracking active — experiment: {experiment}")

## Configure LM

Set your language model backend here.

In [ ]:
lm = dspy.LM(
    "cerebras/qwen-3-235b-a22b-instruct-2507",
    api_key=os.environ["CEREBRAS_API_KEY"],
)
dspy.configure(lm=lm)

## Test: Transform a Legal Article

In [ ]:
AHVG_ART_5 = """\
Art. 5 — Beiträge von Einkommen aus unselbständiger Erwerbstätigkeit

1 Vom Einkommen aus unselbständiger Erwerbstätigkeit, nachfolgend massgebender \
Lohn genannt, wird ein Beitrag von 4,35 Prozent erhoben.

2 Als massgebender Lohn gilt jedes Entgelt für in unselbständiger Stellung auf \
bestimmte oder unbestimmte Zeit geleistete Arbeit.\
"""

transformer = LegalTransformer()
result = transformer(
    legal_article_text=AHVG_ART_5,
    article_reference="AHVG Art. 5",
)

print("=== OpenFisca Variable ===")
print(result.openfisca_variable)
print("\n=== Parameter YAML ===")
print(result.parameter_yaml)
print("\n=== Reasoning ===")
print(result.reasoning)

## Personas

In [ ]:
for p in ALL_PERSONAS:
    print(f"{p.name} ({p.canton}) — {p.description}")

## Execute Generated Code

Run the generated OpenFisca variable against test data to verify it computes correctly.

In [ ]:
# Run generated code with Anna's input data
anna_input = {
    "persons": {
        "anna": {
            "gross_monthly_salary": {"2024-01": 7083.33},
            "age": {"2024-01": 35},
        },
    },
    "households": {"hh": {"parents": ["anna"]}},
}

exec_result = run_generated_code(
    result.openfisca_variable,
    result.parameter_yaml,
    input_data=anna_input,
    period="2024-01",
)

if exec_result.success:
    print("Execution: SUCCESS")
    for var_name, value in exec_result.computed_values.items():
        print(f"  {var_name} = {value:.2f}")
    print(f"\nExpected AHV employee contribution: 308.12 (CHF 7,083.33 × 4.35%)")
else:
    print(f"Execution: FAILED at stage '{exec_result.error_stage}'")
    print(f"  Error: {exec_result.error}")

## Persona Test Results

Run the generated code against all five test personas and check expected values.

In [ ]:
reports = run_persona_tests(result.openfisca_variable, result.parameter_yaml)

for report in reports:
    status = "PASS" if report["passed"] else "FAIL"
    print(f"[{status}] {report['name']}")
    if report["result"].success:
        for var, val in report["result"].computed_values.items():
            print(f"       {var} = {val:.2f}")
    else:
        print(f"       Error ({report['result'].error_stage}): {report['result'].error}")
    if report["mismatches"]:
        for var, (expected, computed) in report["mismatches"].items():
            print(f"       MISMATCH {var}: expected={expected}, computed={computed:.2f}")

passed = sum(1 for r in reports if r["passed"])
print(f"\nPass rate: {passed}/{len(reports)}")